In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

os.makedirs('figures', exist_ok=True)

plt.rcParams.update({
    'font.size':        16,
    'axes.titlesize':   18,
    'axes.labelsize':   16,
    'xtick.labelsize':  14,
    'ytick.labelsize':  14,
    'legend.fontsize':  14,
})

# Raw benchmark results: one row per (device, wraparound, circuit, config, seed)
df  = pd.read_csv('Results/paper.csv')   # SNAIL topologies
dft = pd.read_csv('Results/ibm.csv') # IBM backends: Prague (33q), Brooklyn (65q), Washington (127q)

# Combine device + wraparound flag into a single topology label
df['topo']  = df['device'] + df['wraparound'].map({True: '_wrap', False: ''})
dft['topo'] = dft['device']  # fake_prague, fake_brooklyn, fake_washington

CONFIG_ORDER = ['Standard SABRE', 'FASST', 'Standard MIRAGE', 'FINESSE']
CFG_COLORS   = ['#4C72B0', '#1B6B3A', '#DD8452', '#7B1D1D']

# All 8 SNAIL topologies (4 module types × wrapped/non-wrapped)
TOPO_ORDER = [
    'square_ring', 'square_ring_wrap',
    'square_ring_diag', 'square_ring_diag_wrap',
    'square_ring_full', 'square_ring_full_wrap',
    'pentagon_ring', 'pentagon_ring_wrap',
]
TOPO_LABELS = {
    'square_ring':           '4-qubit, 4-edge module',
    'square_ring_wrap':      '4-qubit, 4-edge module (wrapped)',
    'square_ring_diag':      '4-qubit, 5-edge module',
    'square_ring_diag_wrap': '4-qubit, 5-edge module (wrapped)',
    'square_ring_full':      '4-qubit, 6-edge module',
    'square_ring_full_wrap': '4-qubit, 6-edge module (wrapped)',
    'pentagon_ring':         '5-qubit, 7-edge module',
    'pentagon_ring_wrap':    '5-qubit, 7-edge module (wrapped)',
}

IBM_TOPO_ORDER = ['fake_prague', 'fake_brooklyn', 'fake_washington']
IBM_TOPO_LABELS = {
    'fake_prague':     'Prague (33q, CZ)',
    'fake_brooklyn':   'Brooklyn (65q, CX)',
    'fake_washington': 'Washington (127q, CX)',
}
IBM_TOPO_COLORS = ['#5B8DB8', '#3A7D44', '#B85C38']

# Updated 18-circuit suite: 6 per size band (8–14q, 15–21q, 22–32q)
CIRCUIT_ORDER = [
    # 8–14q
    'qpeexact_n8', 'wstate_n8', 'qft_n10', 'ae_n10', 'ghz_n10', 'vqe_two_local_n10',
    # 15–21q
    'seca_n11', 'multiplier_n15', 'dnn_n16', 'qec9xz_n17', 'square_root_n18', 'bv_n19',
    # 22–32q
    'qft_n24', 'qaoa_n25_p3', 'ising_n26', 'qft_n32', 'qaoa_n32_p3', 'random_n32_d50',
]
CIRC_LABELS = {
    'qpeexact_n8':       'QPE\n(n8)',
    'wstate_n8':         'W-st\n(n8)',
    'qft_n10':           'QFT\n(n10)',
    'ae_n10':            'AE\n(n10)',
    'ghz_n10':           'GHZ\n(n10)',
    'vqe_two_local_n10': 'VQE\n(n10)',
    'seca_n11':          'SECA\n(n11)',
    'multiplier_n15':    'Mult\n(n15)',
    'dnn_n16':           'DNN\n(n16)',
    'qec9xz_n17':        'QEC\n(n17)',
    'square_root_n18':   '√n\n(n18)',
    'bv_n19':            'BV\n(n19)',
    'qft_n24':           'QFT\n(n24)',
    'qaoa_n25_p3':       'QAOA\n(n25)',
    'ising_n26':         'Ising\n(n26)',
    'qft_n32':           'QFT\n(n32)',
    'qaoa_n32_p3':       'QAOA\n(n32)',
    'random_n32_d50':    'Rand\n(n32)',
}

# Each algorithm post-selects its best trial by its own native objective:
#   SABRE   → min SWAP count  (as described in the SABRE paper)
#   MIRAGE  → min depth       (as described in the MIRAGE paper)
#   FASST/FINESSE → min accumulated log-infidelity cost
POST_SELECT = {
    'Standard SABRE':  'swaps',
    'Standard MIRAGE': 'depth',
    'FASST':           'lf_cost',
    'FINESSE':         'lf_cost',
}

def post_select(df, groupby_cols=('topo', 'circuit')):
    """For each (groupby_cols, config) group, keep the single row with the
    lowest value of that config's post-selection column (best trial of N seeds)."""
    frames = []
    for cfg, col in POST_SELECT.items():
        sub = df[df.config == cfg]
        best_idx = sub.groupby(list(groupby_cols))[col].idxmin()
        frames.append(sub.loc[best_idx])
    return pd.concat(frames).reset_index(drop=True)

# Filter: only topologies/circuits actually present in the data
present_topos = [t for t in TOPO_ORDER if t in df['topo'].values]
nowrap_topos  = [t for t in present_topos if not t.endswith('_wrap')]
present_circs = [c for c in CIRCUIT_ORDER if c in df['circuit'].values]
present_ibm   = [t for t in IBM_TOPO_ORDER if t in dft['topo'].values]

COMPARE_CFGS   = [c for c in CONFIG_ORDER if c != 'Standard SABRE']
COMPARE_COLORS = [c for cfg, c in zip(CONFIG_ORDER, CFG_COLORS) if cfg != 'Standard SABRE']

# Apply post-selection
ps_df  = post_select(df[df['topo'].isin(nowrap_topos)])
ps_dft = post_select(dft[dft['topo'].isin(present_ibm)], groupby_cols=('topo', 'circuit'))

print(f'Circuits ({len(present_circs)}): {present_circs}')
print(f'Non-wrapped SNAIL topologies ({len(nowrap_topos)}): {nowrap_topos}')
print(f'IBM topologies: {present_ibm}')
print(f'IBM circuits:   {sorted(dft.circuit.unique())}')

In [ ]:
# ── Figs 1, 2, 3: lf improvement, swap change, depth change per circuit ────────
# → fig_lf_improvement_per_circ.pdf   positive = better lf (improvement vs SABRE)
# → fig_swap_penalty_per_circ.pdf     positive = more SWAPs (penalty vs SABRE)
# → fig_depth_penalty_per_circ.pdf    positive = more depth (penalty vs SABRE)
#
# % calculation per bar:
#   For each circuit × non-wrapped topology pair, compute the % change vs SABRE,
#   then average those 4 per-topology values into one bar per circuit.
#   This gives equal weight to each (topo, circuit) experiment — not inflated by
#   topologies that happen to have larger absolute lf/swap/depth values.
#
#   lf improvement:  sign=+1 → 100 * (SABRE - algo) / SABRE  (positive = algo improved)
#   SWAP/depth penalty: sign=-1 → -100 * (SABRE - algo) / SABRE = 100 * (algo - SABRE) / SABRE
#                                                                   (positive = algo is worse)

circs = [c for c in CIRCUIT_ORDER if c in ps_df['circuit'].values]
xs    = np.arange(len(circs))
n_cfg = len(COMPARE_CFGS)
w     = 0.72 / n_cfg

# (col, y-axis label, title, flip, output filename)
# flip=False → positive = improvement (lf)
# flip=True  → positive = penalty (swaps, depth)
PANELS = [
    ('lf_cost', r'% $\Sigma\,{-}\log F$ improvement vs SABRE',
     r'$\Sigma\,{-}\log F$ improvement vs SABRE',
     False, 'fig_lf_improvement_per_circ.pdf'),
    ('swaps', '% SWAP penalty vs SABRE',
     'SWAP penalty vs SABRE',
     True, 'fig_swap_penalty_per_circ.pdf'),
    ('depth', '% depth penalty vs SABRE',
     'Depth penalty vs SABRE',
     True, 'fig_depth_penalty_per_circ.pdf'),
]

for col, ylabel, title, flip, fname in PANELS:
    sign = -1 if flip else 1  # flips so penalty panels show positive = worse
    fig, ax = plt.subplots(figsize=(max(10, len(circs) * 1.2), 7))
    ax.set_title(title, fontweight='bold')

    for k, (cfg, color) in enumerate(zip(COMPARE_CFGS, COMPARE_COLORS)):
        vals = []
        for circ in circs:
            pcts = []
            for topo in nowrap_topos:
                # Fetch post-selected best row for SABRE and this algorithm
                b = ps_df[(ps_df.topo==topo)&(ps_df.circuit==circ)&(ps_df.config=='Standard SABRE')]
                t = ps_df[(ps_df.topo==topo)&(ps_df.circuit==circ)&(ps_df.config==cfg)]
                # Guard: skip if either row is missing or SABRE baseline is zero
                if len(b) and len(t) and float(b[col].values[0]) > 0:
                    # sign * (SABRE - algo) / SABRE — sign controls improvement vs penalty direction
                    pcts.append(sign * 100 * (float(b[col].values[0]) - float(t[col].values[0])) / float(b[col].values[0]))
            # Average over the 4 non-wrapped topologies for this circuit
            vals.append(np.mean(pcts) if pcts else 0.0)
        ax.bar(xs + (k - (n_cfg-1)/2)*w, vals, w, color=color, label=cfg)

    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xticks(xs)
    ax.set_xticklabels([CIRC_LABELS.get(c, c) for c in circs])
    ax.set_ylabel(ylabel)
    ax.legend(loc='upper right', frameon=False, ncol=2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
    plt.tight_layout()
    plt.savefig(f'figures/{fname}', bbox_inches='tight')
    plt.show()
    print(f'Saved → figures/{fname}')

In [ ]:
# ── Fig: lf cost per topology — FINESSE post-selected, non-wrapped ─────────────
# → fig_topo_lf_finesse.pdf
#
# Shows absolute accumulated log-infidelity (lower = better circuit fidelity),
# NOT a % vs SABRE. One bar group per circuit, one bar per topology.
# Only FINESSE results shown, post-selected by min lf_cost across 20 seeds.
# Non-wrapped topologies only (wraparound=False) to isolate module fidelity effect.

# Re-run post-selection on all non-wrapped topologies (same filter as ps_df)
ps_all_topos = post_select(df[~df['topo'].str.endswith('_wrap')])
# Keep only FINESSE rows and average lf_cost per (topo, circuit)
# (post_select already picks 1 row per group, so mean here is a no-op safety step)
mean_tc = ps_all_topos[ps_all_topos['config'] == 'FINESSE'].groupby(['topo', 'circuit'])[['lf_cost']].mean().reset_index()

circs = [c for c in CIRCUIT_ORDER if c in mean_tc['circuit'].values]
topos = [t for t in TOPO_ORDER if t in mean_tc['topo'].values and not t.endswith('_wrap')]
xs    = np.arange(len(circs))
n_t   = len(topos)
w     = 0.82 / n_t
YMAX  = 15.0  # cap y-axis; annotate bars that exceed it

topo_colors = plt.cm.tab10(np.linspace(0, 0.8, n_t))

fig, ax = plt.subplots(figsize=(max(8, len(circs) * 1.2), 4))
ax.set_title(r'Mean $\Sigma\,{-}\log F$ per topology — FINESSE', fontweight='bold')

overflow_rows = []
for k, (topo, color) in enumerate(zip(topos, topo_colors)):
    vals = []
    for c in circs:
        r = mean_tc[(mean_tc['topo'] == topo) & (mean_tc['circuit'] == c)]
        vals.append(float(r['lf_cost'].values[0]) if len(r) else 0.0)
    # Clip bars at YMAX and annotate overflow values above the cap
    bar_vals = [min(v, YMAX) for v in vals]
    ax.bar(xs + (k - (n_t-1)/2)*w, bar_vals, w, color=color, label=TOPO_LABELS.get(topo, topo))
    for i, v in enumerate(vals):
        if v > YMAX:
            ax.text(xs[i] + (k - (n_t-1)/2)*w, YMAX + 0.2, f'{v:.1f}',
                    ha='center', va='bottom', fontsize=9, rotation=90)
            overflow_rows.append({'topology': TOPO_LABELS.get(topo, topo),
                                  'circuit': circs[i], 'lf_cost': v})

ax.set_ylim(0, YMAX + 1.5)
ax.set_xticks(xs)
ax.set_xticklabels([CIRC_LABELS.get(c, c) for c in circs])
ax.legend(loc='upper left', frameon=False, ncol=2, fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
plt.tight_layout()
plt.savefig('figures/fig_topo_lf_finesse.pdf', bbox_inches='tight')
plt.show()
print('Saved → figures/fig_topo_lf_finesse.pdf')
if overflow_rows:
    print('\nValues exceeding y=15:')
    print(pd.DataFrame(overflow_rows).to_string(index=False))

In [ ]:
# ── Fig: IBM lf improvement — FINESSE, grouped by backend ────────────────────
# → fig_ibm_lf_improvement.pdf
#
# One bar group per circuit, one bar per IBM backend (Prague/Brooklyn/Washington).
# % improvement: 100 * (SABRE_lf - FINESSE_lf) / SABRE_lf  (positive = FINESSE better).
# ps_dft post-selected: SABRE by min_swaps, FINESSE by min_lf_cost, per (topo, circuit).

circs_t  = [c for c in CIRCUIT_ORDER if c in ps_dft['circuit'].values]
ibm_topos = [t for t in IBM_TOPO_ORDER if t in ps_dft['topo'].values]
xs = np.arange(len(circs_t))
n_t = len(ibm_topos)
w   = 0.72 / n_t

fig, ax = plt.subplots(figsize=(max(10, len(circs_t) * 1.4), 5))
ax.set_title(r'$\Sigma\,{-}\log F$ improvement vs SABRE — IBM backends', fontweight='bold')

finesse_color_ibm = IBM_TOPO_COLORS

for k, (topo, color) in enumerate(zip(ibm_topos, finesse_color_ibm)):
    vals = []
    for c in circs_t:
        base = ps_dft[(ps_dft['topo'] == topo) & (ps_dft['circuit'] == c) & (ps_dft['config'] == 'Standard SABRE')]['lf_cost']
        tgt  = ps_dft[(ps_dft['topo'] == topo) & (ps_dft['circuit'] == c) & (ps_dft['config'] == 'FINESSE')]['lf_cost']
        if len(base) and len(tgt) and float(base.values[0]) > 0:
            vals.append(100 * (float(base.values[0]) - float(tgt.values[0])) / float(base.values[0]))
        else:
            vals.append(0.0)
    ax.bar(xs + (k - (n_t-1)/2)*w, vals, w, color=color, label=IBM_TOPO_LABELS[topo])

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(xs)
ax.set_xticklabels([CIRC_LABELS.get(c, c) for c in circs_t])
ax.set_ylabel(r'% $\Sigma\,{-}\log F$ improvement vs SABRE')
ax.legend(loc='upper right', frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
plt.tight_layout()
plt.savefig('figures/fig_ibm_lf_improvement.pdf', bbox_inches='tight')
plt.show()
print('Saved → figures/fig_ibm_lf_improvement.pdf')

In [ ]:
# ── Headline numbers (verify against paper text) ───────────────────────────────
# Conventions: lf improvement = positive good, SWAP/depth penalty = positive bad

print('=== SNAIL (non-wrapped, per (topo,circuit) averaged) ===')
print()
for cfg in COMPARE_CFGS:
    lf_pcts, sw_pcts, d_pcts = [], [], []
    for topo in nowrap_topos:
        for circ in present_circs:
            b = ps_df[(ps_df.topo==topo)&(ps_df.circuit==circ)&(ps_df.config=='Standard SABRE')]
            t = ps_df[(ps_df.topo==topo)&(ps_df.circuit==circ)&(ps_df.config==cfg)]
            if len(b) and len(t):
                if float(b['lf_cost'].iloc[0]) > 0:
                    lf_pcts.append(100*(float(b['lf_cost'].iloc[0])-float(t['lf_cost'].iloc[0]))/float(b['lf_cost'].iloc[0]))
                if float(b['swaps'].iloc[0]) > 0:
                    sw_pcts.append(100*(float(t['swaps'].iloc[0])-float(b['swaps'].iloc[0]))/float(b['swaps'].iloc[0]))
                if float(b['depth'].iloc[0]) > 0:
                    d_pcts.append(100*(float(t['depth'].iloc[0])-float(b['depth'].iloc[0]))/float(b['depth'].iloc[0]))
    print(f'{cfg}')
    print(f'  lf improvement: {np.mean(lf_pcts):+.1f}%')
    print(f'  SWAP penalty:   {np.mean(sw_pcts):+.1f}%')
    print(f'  depth penalty:  {np.mean(d_pcts):+.1f}%')
    print()

print('=== IBM backends (per (backend, circuit) averaged) ===')
print()
ibm_circs = [c for c in CIRCUIT_ORDER if c in ps_dft['circuit'].values]
for topo in present_ibm:
    print(f'--- {IBM_TOPO_LABELS[topo]} ---')
    for cfg in COMPARE_CFGS:
        lf_pcts, sw_pcts = [], []
        for circ in ibm_circs:
            b = ps_dft[(ps_dft.topo==topo)&(ps_dft.circuit==circ)&(ps_dft.config=='Standard SABRE')]
            t = ps_dft[(ps_dft.topo==topo)&(ps_dft.circuit==circ)&(ps_dft.config==cfg)]
            if len(b) and len(t):
                if float(b['lf_cost'].iloc[0]) > 0:
                    lf_pcts.append(100*(float(b['lf_cost'].iloc[0])-float(t['lf_cost'].iloc[0]))/float(b['lf_cost'].iloc[0]))
                if float(b['swaps'].iloc[0]) > 0:
                    sw_pcts.append(100*(float(t['swaps'].iloc[0])-float(b['swaps'].iloc[0]))/float(b['swaps'].iloc[0]))
        print(f'  {cfg}: lf {np.mean(lf_pcts):+.1f}%  SWAP {np.mean(sw_pcts):+.1f}%')
    print()